# 검색 베이스라인 돌려보기

질문 하나가 ① 쪼개기 → ② 분류 → ③ 라우터 → ④ 검색 → `LlmRequest` 까지 가는 과정을 단계별로 봐요.

**Colab에서**: 런타임 → 런타임 유형 변경 → GPU(L4/T4) 선택 후 위에서부터 차례로 실행하면 돼요.
- 첫 셀이 저장소 3개를 `/content` 에 받아요: `RAG`(이 노트북 + `rag` 패키지), `DB`(규정·법령 조각 파일), `goldenset`(데이터 형식·라우터)
- DB(docker)는 필요 없어요. 검색 대상은 `DB/data/regs`, `DB/data/laws` 의 `chunks.jsonl` 을 바로 읽어요.
- 처음 한 번은 BGE-M3 다운로드(약 2.3GB) + 조각 약 2,600개 임베딩으로 몇 분 걸려요. 임베딩은 `RAG_CACHE_DIR` 에 저장돼서 다음부터는 바로 불러와요.

In [ ]:
import os, subprocess, sys
from pathlib import Path

BASE = Path("/content") if Path("/content").exists() else Path.cwd().parent
ORG = "https://github.com/Jaeyong-s-HR-Chatbot"
for name in ["RAG", "DB", "goldenset"]:
    d = BASE / name
    if d.exists():
        subprocess.run(["git", "-C", str(d), "pull", "-q"], check=False)   # 최신으로
    else:
        subprocess.run(["git", "clone", "-q", f"{ORG}/{name}", str(d)], check=True)

RAG_DIR = BASE / "RAG"          # 드라이브에 올린 코드를 쓰려면 여기를 그 경로로 바꾸세요
assert (RAG_DIR / "rag" / "__init__.py").is_file(), f"{RAG_DIR}/rag 패키지가 없어요. RAG 저장소에 rag/ 폴더가 올라가 있는지 확인하세요."

os.environ["DB_DIR"] = str(BASE / "DB")
os.environ["GOLDENSET_DIR"] = str(BASE / "goldenset")
os.environ.setdefault("RAG_CACHE_DIR", str(BASE / "rag_cache"))   # 드라이브에 두면 런타임이 바뀌어도 재사용: "/content/drive/MyDrive/rag_cache"
# os.environ["RAG_EMBEDDER"] = "mock"   # GPU 없이 코드 흐름만 볼 때 (점수는 의미 없음)

os.chdir(RAG_DIR)
if str(RAG_DIR) not in sys.path:
    sys.path.insert(0, str(RAG_DIR))
print("작업 폴더:", Path.cwd())

In [ ]:
!pip install -q -r requirements.txt
!pip install -q kiwipiepy || echo "kiwipiepy 설치 실패 → BM25는 어절+2-gram 으로 대신해요"

In [ ]:
from rag import Pipeline, load_config, split_question

cfg = load_config()          # 기본 구성 C
pipe = Pipeline(cfg)         # BGE-M3 불러오기 + 임베딩(처음만) + BM25 색인
print("임베딩 장치:", pipe.embedder.device, "| 검색 대상 조각 수:", len(pipe.retriever.rows))

## ① 질문 쪼개기

In [ ]:
for q in ["결혼하면 휴가 며칠이고 경조금 얼마야?",
          "5일 썼는데, 며칠까지 돼요?",
          "복지포인트는 언제 들어오고 어디서 쓸 수 있어요?"]:
    print(q, "→", split_question(q))

## ②③ 분류기와 라우터

In [ ]:
q = "결혼하면 휴가 며칠이고 경조금 얼마야?"
r = pipe.run(q)
print("상태:", r.status, r.message or "")
for s in r.subs:
    print(s.question)
    print("  카테고리:", [(c, round(x, 3)) for c, x in s.query_class.categories], "confident =", s.query_class.confident)
    print("  전체 점수:", {c: round(x, 3) for c, x in sorted(s.query_class.scores.items(), key=lambda kv: -kv[1])})
    print("  slots:", s.query_class.slots, "| 라우터:", s.route)

## ④ 검색 후보 (최대 20개)

In [ ]:
import pandas as pd

s = r.subs[0]
print("필터:", s.search.used_filter, s.search.filter_categories, "| fallback:", s.search.used_fallback,
      "| 1등 코사인:", s.search.top_cos, "|", s.search.doc_kind_counts)
df = pd.DataFrame([c.to_dict() for c in s.search.candidates])
df.insert(1, "title", [pipe.retriever.rows[c]["doc_name"] + " " + (pipe.retriever.rows[c]["article"] or "") for c in df.chunk_id])
df

## LLM 에 넘길 것: ChunkHit 8개 → LlmRequest

In [ ]:
from rag.goldenset import build_messages

for c in r.chunks:
    print(c.rank, c.chunk_id, c.title, round(c.score, 3), c.categories)
print()
print(build_messages(r.llm_request)[1]["content"][:800])

## 실험 구성 바꿔 보기 (A0 / A1 / B / C)

모델과 색인은 그대로 두고 스위치만 바꿔요.

| 구성 | 검색 |
|---|---|
| A0 | Dense만 |
| A1 | + BM25, RRF로 합치기 |
| B | + 분류 결과로 하드 필터 |
| C | + confidence 따라 하드/소프트 필터, 약하면 필터 해제 (기본) |

In [ ]:
q = "서울 출장 가면 숙박비 얼마까지 나와?"
for name in ["A0", "A1", "B", "C"]:
    rr = pipe.with_config(cfg.preset(name)).run(q)
    print(name, [c.chunk_id for c in rr.chunks[:5]])

## 범위 밖 · 애매함 · 팀 단위

In [ ]:
for q in ["요즘 주식 뭐 사야 돼?", "휴가 며칠이야?", "우리 팀 연차 현황 보여줘", "나 올해 연차 몇 개야?"]:
    rr = pipe.run(q)
    print(f"{q} → {rr.status} | hr_needed={rr.hr_needed} | {rr.message or ''}")